<a href="https://colab.research.google.com/github/yc386/paasta_2026_denovo/blob/main/paasta_denovo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/PAASTA-community/paasta-community.github.io/blob/master/assets/media/paasta_no_bg.png?raw=1" width="180" height="85">
<img src="https://github.com/yc386/forked_pasta/blob/master/assets/media/summer_school.png?raw=1" width="460" height="80">


# *De novo* sequencing: from a PRIDE `.raw` file to annotated spectra

This is a 45-minute practical that walks you through the following:

1. **Fetch** a Thermo `.raw` file from the PRIDE FTP archive
2. **Convert** `.raw` → `mzML` with **ThermoRawFileParser**
3. **Parse** the `mzML` into a `parquet` for **InstaNovo** CLI and a `mgf` for their Hugging Face space
4. ***De Novo*** the spectra with **InstaNovo**, refined by **InstaNovo+**
5. **Inspect** predictions and **annotate** spectra with its predicted b/y ions using **spectrum_utils**
6. **Score** predictions with label-free quality **metrics** (confidence & ion coverage)
7. *(optional, advanced)* **Calibrate** confidence and control **FDR** with **winnow**

> **Before you start:** set the GPU runtime — *Runtime → Change runtime type → Hardware accelerator → GPU (T4)*

>A Google account may be needed to run notebook cells. You may also click *File → Save a copy in Drive*.

>Our working directory(pwd) is /content/ (default), so no need to mount your own google drive unless you'd like to.

## Step 0 — Setup (run once; the runtime will restart)
**Don't click the prompt when Google asks you to restart, it will be done automatically!**
> When the runtime reconnects you'll see a **"Your session crashed for an unknown reason"** banner — don't be alarmed, that's just os.kill & restarting. The pip install cell is **safe to re-run** (it skips automatically if InstaNovo is already installed); just continue from the GPU check below.

In [ ]:
#@title pip install cell (takes c.5 mins; the runtime will restart automatically)
# Safe to re-run: if InstaNovo already imports, the whole setup is skipped.
try:
    import instanovo  # noqa: F401
    print("InstaNovo already installed - skipping setup. Continue from the GPU check below.")
except ImportError:
    # Install InstaNovo from GitHub main: the PyPI 1.2.2 release fails to load the
    # InstaNovo+ diffusion checkpoint (torch weights_only=True). Fixed by PR #140
    # (commit 43b3066). Revert to "instanovo[cu126]>=1.2.x" once that's released.
    !pip install biopython "instanovo[cu126] @ git+https://github.com/instadeepai/InstaNovo.git@main" pyteomics psims lxml polars pyarrow spectrum_utils

    # Colab pre-installs its own torch build, which clashes with the CUDA 12.6
    # wheels instanovo[cu126] needs. Force a known-good, matching CUDA 12.6 set.
    !pip uninstall -y torch torchvision torchaudio
    !pip install --no-cache-dir torch==2.6.0+cu126 torchvision==0.21.0+cu126 torchaudio==2.6.0+cu126 --index-url https://download.pytorch.org/whl/cu126

    print("Installation complete. Restarting runtime to apply changes...")
    import os
    os.kill(os.getpid(), 9)

In [30]:
#@title >>> After the runtime restarts, start running from **HERE**. Check GPU available for InstaNovo[cu126]<<<
!nvidia-smi -L 2>/dev/null || echo "No GPU -> Runtime > Change runtime type > GPU"
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
# Pin down exactly which InstaNovo / InstaNovo+ versions we're running (helps debugging)
!instanovo version

No GPU -> Runtime > Change runtime type > GPU
torch: 2.6.0+cu126 | CUDA available: True
┏━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Package    ┃ Version ┃
┡━━━━━━━━━━━━╇━━━━━━━━━┩
│ InstaNovo  │ 1.2.2   │
│ InstaNovo+ │ 1.2.2   │
│ NumPy      │ 2.4.6   │
│ PyTorch    │ 2.12.0  │
└────────────┴─────────┘


## Step 1 — Fetch a .raw file from PRIDE

Every PRIDE project has a predictable FTP path:

```
https://ftp.pride.ebi.ac.uk/pride/data/archive/<year>/<month>/<accession>/
```
We are going to download a small file from [PXD058287](https://ftp.pride.ebi.ac.uk/pride/data/archive/2025/09/PXD058287/):
```
20240425-2151_QEHF3_1007093_ONJ_TR_MC_BLG_LS_T12_3.raw (179.7 MB, downloading time: around 1 min)
```

In [31]:
import os, requests

ACCESSION = "PXD058287"
YEAR, MONTH = "2025", "09"
RAW_NAME = "20240425-2151_QEHF3_1007093_ONJ_TR_MC_BLG_LS_T12_3.raw"
URL = f"https://ftp.pride.ebi.ac.uk/pride/data/archive/{YEAR}/{MONTH}/{ACCESSION}/{RAW_NAME}"
RAW_PATH = RAW_NAME

if not os.path.exists(RAW_PATH):
    print("Downloading", URL)
    with requests.get(URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(RAW_PATH, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
print("RAW size (MB):", round(os.path.getsize(RAW_PATH) / 1e6, 1))

RAW size (MB): 179.7


## Step 2 — Convert Thermo .raw → mzML
`.raw` is Thermo's vendor specific (closed) format. **[ThermoRawFileParser](https://github.com/compomics/ThermoRawFileParser)** is an open-source parser that runs on all platforms to convert `.raw` to open formats: `mzML`/`MGF`. We are using `v.2.0.0-dev` without `Mono` dependency.
> -f=0|1|2|3|4, 0 for `MGF`, 1 for `mzML`(plain), 2 for `indexed mzML`

In [32]:
%%bash
# Skip the download/unzip if ThermoRawFileParser is already present.
if [ -f ThermoRawFileParser/ThermoRawFileParser ]; then
    echo "ThermoRawFileParser already present - skipping download."
else
    wget -q https://github.com/CompOmics/ThermoRawFileParser/releases/download/v.2.0.0-dev/ThermoRawFileParser-v.2.0.0-dev-linux.zip
    unzip -q ThermoRawFileParser-v.2.0.0-dev-linux.zip -d ThermoRawFileParser && rm ThermoRawFileParser-v.2.0.0-dev-linux.zip
    echo "ThermoRawFileParser downloaded."
fi

ThermoRawFileParser already present - skipping download.


In [33]:
import os

# Reuse RAW_PATH from Step 1 so the filename lives in exactly one place.
# ThermoRawFileParser writes the .mzML next to the input with the same base name.
MZML_PATH = RAW_PATH.replace(".raw", ".mzML")
fmt = 2  # 0=MGF, 1=mzML (plain), 2=indexed mzML

# Skip the conversion if the .mzML already exists.
if os.path.exists(MZML_PATH):
    print(f"{MZML_PATH} already exists - skipping conversion.")
else:
    !ThermoRawFileParser/ThermoRawFileParser -i "{RAW_PATH}" -f "{fmt}" -z

20240425-2151_QEHF3_1007093_ONJ_TR_MC_BLG_LS_T12_3.mzML already exists - skipping conversion.


## Step 3 — Parse the mzML
> We are going to see what's inside a `mzML` by using InstaNovo's `SpectrumDataFrame`. `.parquet` is like a `.csv` but smaller to store and faster for a computer to process.
> We then write a **higher-quality subset of 100 spectra** to both `.parquet` (for the CLI) and `.mgf` (for the Hugging Face space). Rather than the first 100 scans — which elute early and are mostly low quality — we keep **charge 2–3** precursors and the **100 most intense** spectra, which give far more confident *de novo* predictions.

In [34]:
import glob
import pandas as pd
from instanovo.utils.data_handler import SpectrumDataFrame

In [36]:
sdf1 = SpectrumDataFrame.load(MZML_PATH, lazy=False, is_annotated=False)
df = sdf1.to_pandas()
print(f"Parsed {len(df)} spectra | columns: {list(df.columns)}")
df.head()  # show a preview, not all rows (the mz/intensity arrays are large)

XMLSyntaxError: Premature end of data in tag spectrumList line 60, line 390242, column 20 (20240425-2151_QEHF3_1007093_ONJ_TR_MC_BLG_LS_T12_3.mzML, line 390242)

In [ ]:
import numpy as np

# The first scans elute early and are mostly low quality (sparse spectra).
# Pick a higher-quality subset instead: precursor charge 2-3 (best for de novo)
# and the 100 most intense spectra (highest total ion current, TIC).
tic  = df["intensity_array"].map(lambda a: float(np.sum(a)))
mask = df["precursor_charge"].isin([2, 3])
df1  = (df[mask].assign(_tic=tic[mask])
                .sort_values("_tic", ascending=False)
                .head(100)
                .drop(columns="_tic")
                .reset_index(drop=True))
df1.to_parquet("head_100.parquet")
print(f"selected {len(df1)} spectra (charge 2-3, top TIC) | median peaks/spectrum: {int(df1['mz_array'].map(len).median())}")

In [ ]:
from pyteomics import mgf
import numpy as np

def df_to_mgf(df, out_path):
    spectra = []
    for _, row in df.iterrows():
        params = {
            "title":       f"{row['experiment_name']}:scan:{int(row['scan_number'])}",
            "pepmass":     float(row["precursor_mz"]),
            "charge":      f"{int(row['precursor_charge'])}+",
            "rtinseconds": float(row["retention_time"]) * 60.0,
            "scans":       int(row["scan_number"]),
        }
        seq = row.get("sequence")
        if seq is not None and str(seq).strip() not in ("", "nan", "None"):
            params["seq"] = str(seq)

        spectra.append({
            "m/z array":       np.asarray(row["mz_array"],        dtype=float),
            "intensity array": np.asarray(row["intensity_array"], dtype=float),
            "params":          params,
        })

    mgf.write(spectra, out_path, file_mode="w")
    return out_path

In [ ]:
df_to_mgf(df1, "head_100.mgf")

## Step 4 — Run InstaNovo for *de novo*
**[InstaNovo](https://github.com/instadeepai/InstaNovo)** is a transformer model that is well-annotated and easy to run. This time we run it **with diffusion refinement** by the **InstaNovo+** model (the CLI default, `--with-refinement`).

When refinement is on, the output `preds.csv` keeps **both** predictions per spectrum:
- `predictions` / `log_probs` — the **refined** (InstaNovo+) sequence and its confidence
- `instanovo_predictions` / `instanovo_prediction_log_probability` — the **original transformer** sequence and confidence

so we can see what refinement changed.
>
- option A: For Google account users, you may run it here using the CLI (command line interface).
- option B: Go to InstaNovo [Hugging Face space](https://huggingface.co/spaces/InstaDeepAI/InstaNovo) and run the `head_100.mgf` we made in Step 3 online. <br />(note **not the `.mzML`** since it may be too big)

In [ ]:
!instanovo predict --help

In [ ]:
# --with-refinement runs the transformer AND refines with InstaNovo+ (diffusion).
# First run downloads the InstaNovo+ checkpoint; ~30-60 s for 100 spectra on a T4.
!instanovo predict --data-path head_100.parquet --output-path preds.csv --denovo --with-refinement

In [ ]:
#@title What did InstaNovo+ refinement change?
import ast
import pandas as pd
import numpy as np

preds = pd.read_csv("preds.csv")

def _tok_to_seq(x):
    # InstaNovo stores the transformer prediction as a list-repr string, e.g. "['P', 'E', ...]"
    if isinstance(x, str) and x.strip().startswith("["):
        return "".join(ast.literal_eval(x))
    return "" if pd.isna(x) else str(x)

preds["transformer"] = preds["instanovo_predictions"].map(_tok_to_seq)
preds["refined"]     = preds["predictions"].astype(str)

changed = preds["transformer"] != preds["refined"]
print(f"InstaNovo+ refinement changed {int(changed.sum())}/{len(preds)} sequences\n")
print(preds.loc[changed, ["scan_number", "transformer", "refined"]].head(8).to_string(index=False))

## Step 5 — Inspect predictions and annotate spectra
> We will pick *de novo* predictions from a list of top and bottom 10 spectra (sorted by model confidence), and visualise them using mirrored plots to see if observed b/y ions matched predicted ones. Generally, higher the ion coverage (observed/predicted ions), the better.

> The mirror plots below use the **refined** (`predictions`) sequence. This is *real* data with **no ground-truth peptide labels**, so we can't compute recall/precision — we use confidence and b/y ion coverage as quality proxies instead (quantified in Step 6).

In [ ]:
import matplotlib.pyplot as plt
import spectrum_utils.spectrum as sus
import spectrum_utils.plot as sup
from spectrum_utils import proforma
from spectrum_utils.fragment_annotation import get_theoretical_fragments
%matplotlib inline

In [ ]:
PARQUET = "head_100.parquet"
#change fragment ion tolerance here
TOL, TOL_MODE = 10, "ppm"

preds   = pd.read_csv("preds.csv")
spectra = pd.read_parquet(PARQUET)

seq_col  = next((c for c in ["predictions","prediction","sequence","peptide"] if c in preds.columns), preds.columns[0])
conf_col = next((c for c in ["log_probs","log_probabilities","confidence"] if c in preds.columns), None)
if conf_col is not None:
    preds["model_confidence"] = np.clip(np.exp(preds[conf_col]), 0, 1)

cand = preds.sort_values(conf_col, ascending=False) if conf_col else preds
cols = ["scan_number", seq_col, conf_col, "model_confidence"]

print("==TOP 10 (most confident)==")
print(cand[cols].head(10).to_string(index=False))

print("\n==BOTTOM 10 (least confident)==")
print(cand[cols].tail(10).to_string(index=False))

def show_mirror(scan_number=None):
    """Pass a scan_number from the list above; None = highest confidence."""
    if scan_number is None:
        row = cand.iloc[0]
    else:
        sel = cand[cand["scan_number"] == scan_number]
        if sel.empty:
            print(f"scan {scan_number} not found in predictions - pick one from the lists above.")
            return
        row = sel.iloc[0]
    peptide, scan = str(row[seq_col]), int(row["scan_number"])
    sp = spectra.loc[spectra["scan_number"] == scan].iloc[0]
    pmz, z = float(sp["precursor_mz"]), int(sp["precursor_charge"])

    obs = sus.MsmsSpectrum(str(scan), pmz, z,
                           np.asarray(sp["mz_array"], float), np.asarray(sp["intensity_array"], float))
    obs = (obs.set_mz_range(100, 1500).remove_precursor_peak(TOL, TOL_MODE)
              .filter_intensity(0.01, 50).scale_intensity("root")
              .annotate_proforma(peptide, TOL, TOL_MODE, ion_types="by"))

    proteoform = proforma.parse(peptide)[0]
    frags = get_theoretical_fragments(proteoform, ion_types="by", max_charge=1)
    frag_mz = np.array([mz for _, mz in frags])
    theo = sus.MsmsSpectrum(f"{peptide} (theoretical)", pmz, z,
                            frag_mz, np.full(len(frag_mz), obs.intensity.max()))
    theo = theo.annotate_proforma(peptide, TOL, TOL_MODE, ion_types="by")

    fig, ax = plt.subplots(figsize=(12, 6))
    sup.mirror(obs, theo, ax=ax)

    # Restyle the lower (theoretical, negative-y) half of the mirror plot as
    # dashed lines. This pokes at matplotlib artists directly, so it may need
    # updating if spectrum_utils / matplotlib change how they draw peaks.
    for line in ax.lines:
        yd = line.get_ydata()
        if len(yd) and np.min(yd) < -1e-9:
            line.set_linestyle("--")
    for coll in ax.collections:
        segs = coll.get_segments()
        if segs and min(s[:, 1].min() for s in segs) < -1e-9:
            coll.set_linestyle("--")

    ax.set_title(f"scan {scan}  |  {peptide}   (top: observed   bottom: theoretical)")
    plt.tight_layout(); plt.show()

# highest-confidence prediction
show_mirror()

In [ ]:
show_mirror(358)

In [ ]:
preds

## Step 6 — Quality metrics without ground truth
This is real data with **no known peptides**, so we can't compute recall/precision. Instead we use two **label-free** signals:
- **model confidence** = `exp(log_probs)` of the (refined) prediction — how sure the model is.
- **ion coverage** = fraction of the predicted peptide's theoretical **b/y** ions actually observed in the spectrum — how well the prediction explains the data.

A trustworthy identification has **both** high confidence *and* high ion coverage.
> Because we selected a higher-quality subset in Step 3 (charge 2–3, most intense), many predictions here have good confidence *and* coverage. Watch for the exceptions: **high confidence with low coverage is a red flag**, and is exactly what confidence-calibration / FDR-control tools (e.g. [winnow](https://github.com/instadeepai/winnow)) are built to catch.

In [ ]:
#@title Confidence & ion coverage (reuses spectrum_utils imports from Step 5)
preds   = pd.read_csv("preds.csv")
spectra = pd.read_parquet("head_100.parquet")
TOL = 10  # ppm, fragment match tolerance

preds["model_confidence"] = np.clip(np.exp(preds["log_probs"]), 0, 1)

def ion_coverage(peptide, mz_array, tol_ppm=TOL):
    """Fraction of theoretical singly-charged b/y ions observed in the spectrum."""
    try:
        prot = proforma.parse(str(peptide))[0]
        frags = get_theoretical_fragments(prot, ion_types="by", max_charge=1)
    except Exception:
        return np.nan
    obs = np.asarray(mz_array, float)
    if not frags or obs.size == 0:
        return np.nan
    matched = sum(np.min(np.abs(obs - t)) / t * 1e6 <= tol_ppm for _, t in frags)
    return matched / len(frags)

mz_by_scan = {int(r["scan_number"]): r["mz_array"] for _, r in spectra.iterrows()}
preds["ion_coverage"] = [ion_coverage(p, mz_by_scan.get(int(s)))
                         for p, s in zip(preds["predictions"], preds["scan_number"])]

print("model confidence : mean %.2f  median %.2f" % (preds["model_confidence"].mean(), preds["model_confidence"].median()))
print("ion coverage     : mean %.2f  median %.2f" % (preds["ion_coverage"].mean(), preds["ion_coverage"].median()))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(preds["model_confidence"], bins=20); ax[0].set(title="Model confidence", xlabel="exp(log_probs)", ylabel="# spectra")
ax[1].hist(preds["ion_coverage"].dropna(), bins=20); ax[1].set(title="Ion coverage (b/y)", xlabel="fraction observed")
ax[2].scatter(preds["model_confidence"], preds["ion_coverage"], alpha=0.6); ax[2].set(title="Confidence vs coverage", xlabel="model confidence", ylabel="ion coverage")
plt.tight_layout(); plt.show()

### Does refinement help?
InstaNovo+ (diffusion) refines the transformer's sequence. Without ground truth, we judge "better" by **ion coverage** (does the new sequence explain more peaks?). The cell below compares transformer vs refined.

> On this dataset you'll typically see refinement **change ~25% of sequences and sharply raise confidence, while mean ion coverage barely moves** — many edits are isobaric (e.g. `L`↔`I`, same mass) or reorderings that don't change which b/y ions match. So here refinement mostly *re-scores* rather than visibly improving spectral explanation.
>
> ⚠️ Ion coverage can't see isobaric fixes, and we have no ground truth — so this is *not* proof refinement is useless (the InstaNovo paper shows clear gains against labelled benchmarks). **winnow can't settle it either**: it calibrates the *transformer's* beam-search output, and the refined sequences have no beams, so it isn't built to compare the two. To truly measure refinement accuracy you need a labelled dataset.

In [ ]:
#@title Did InstaNovo+ refinement improve the predictions?
# Compare the transformer vs the refined sequence on the same label-free metrics.
preds["transformer_seq"] = preds["instanovo_predictions"].map(_tok_to_seq)
preds["t_coverage"]   = [ion_coverage(p, mz_by_scan.get(int(s)))
                         for p, s in zip(preds["transformer_seq"], preds["scan_number"])]
preds["t_confidence"] = np.clip(np.exp(preds["instanovo_prediction_log_probability"]), 0, 1)

changed = preds["transformer_seq"] != preds["predictions"].astype(str)
print(f"refinement changed {int(changed.sum())}/{len(preds)} sequences")
print("mean confidence  : transformer %.3f -> refined %.3f" % (preds["t_confidence"].mean(), preds["model_confidence"].mean()))
print("mean ion coverage: transformer %.3f -> refined %.3f" % (preds["t_coverage"].mean(), preds["ion_coverage"].mean()))

sub = preds[changed]
imp = int((sub["ion_coverage"] > sub["t_coverage"] + 1e-9).sum())
wor = int((sub["ion_coverage"] < sub["t_coverage"] - 1e-9).sum())
print(f"among the {len(sub)} changed: ion coverage improved {imp}, worsened {wor}, unchanged {len(sub) - imp - wor}")

In [ ]:
#@title Per-residue confidence (transformer model)
# The diffusion-refined output has no per-token probabilities, so we plot the
# TRANSFORMER prediction's per-residue confidence.
import ast

def show_residue_confidence(scan_number):
    row = preds.loc[preds["scan_number"] == scan_number]
    if row.empty:
        print(f"scan {scan_number} not found - pick one from the lists above.")
        return
    row = row.iloc[0]
    tokens = [t.strip().strip("'\"") for t in row["instanovo_predictions"].strip("[]").split(",")]
    tlp = np.asarray(ast.literal_eval(row["instanovo_prediction_token_log_probabilities"]), float)
    conf = np.exp(tlp[:len(tokens)])
    fig, ax = plt.subplots(figsize=(max(6, 0.5 * len(tokens)), 3))
    ax.bar(range(len(tokens)), conf, color="#4C72B0")
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("residue confidence")
    ax.set_title(f"scan {scan_number}: {''.join(tokens)}  (transformer)")
    plt.tight_layout(); plt.show()

# most confident transformer prediction
top = preds.assign(_c=np.exp(preds["instanovo_prediction_log_probability"])).sort_values("_c", ascending=False)
show_residue_confidence(int(top.iloc[0]["scan_number"]))

## Step 7 (optional, advanced) — Calibrate confidence & control FDR with winnow
The model's `log_probs` confidence isn't *calibrated* — a score of 0.9 doesn't literally mean "90% likely correct". [**winnow**](https://github.com/instadeepai/winnow) recalibrates *de novo* confidence with a **pretrained general model** and estimates a **false discovery rate (FDR)** *without ground-truth labels*, so you can keep only PSMs below a chosen FDR.

> **Heads-up:**
> - Needs **internet**: it downloads a pretrained calibrator from Hugging Face and queries the public **[Koina](https://koina.wilhelmlab.org)** server for fragment-ion predictions.
> - We install winnow from its **`revisions`** branch for now (the released PyPI build can't yet load the current safetensors model). **Revert to `pip install winnow-fdr` once the open PRs are merged.**
> - Run this **last** — the winnow install may change `torch`. Our InstaNovo predictions are already saved to CSV, so that's fine.

In [ ]:
#@title Install winnow (confidence calibration & FDR control)
# Installing from the `revisions` branch: the published winnow-general-model
# (safetensors format) needs loader updates not yet in the PyPI release.
# TODO: revert to `!pip install -q winnow-fdr` once the outstanding PRs are merged.
!pip install -q "winnow-fdr @ git+https://github.com/instadeepai/winnow.git@revisions"

In [ ]:
# winnow calibrates the transformer's beam-search output, so make a
# transformer-only prediction (keeps the `predictions_beam_*` columns winnow needs).
!instanovo predict --data-path head_100.parquet --output-path preds_transformer.csv --denovo --no-refinement

In [ ]:
#@title Load predictions into winnow, then calibrate confidence
import os, warnings, winnow
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from hydra import initialize_config_dir, compose
from hydra.utils import instantiate
from winnow.calibration.calibrator import ProbabilityCalibrator
from winnow.fdr.nonparametric import NonParametricFDRControl

# winnow merges predictions with spectra on spectrum_id (= experiment_name:scan_number)
sp = pd.read_parquet("head_100.parquet")
sp["spectrum_id"] = sp["experiment_name"].astype(str) + ":" + sp["scan_number"].astype(str)
sp.to_parquet("head_100_winnow.parquet")

# Build the InstaNovo data loader from winnow's packaged config
WINNOW_CFG = os.path.join(os.path.dirname(winnow.__file__), "configs")
with initialize_config_dir(config_dir=WINNOW_CFG, version_base="1.3", job_name="paasta"):
    cfg = compose(config_name="train")
loader = instantiate(cfg.data_loader)
dataset = loader.load(data_path="head_100_winnow.parquet", predictions_path="preds_transformer.csv")
print("loaded", len(dataset), "PSMs")

# Pretrained general calibrator — no training or labels needed
calibrator = ProbabilityCalibrator.load()  # downloads from HuggingFace
# Fragment-match features query the public Koina server (needs internet)
calibrator.apply_koina_server_overrides(server_url="koina.wilhelmlab.org:443", ssl=True)
calibrator.compute_features(dataset)
calibrator.predict(dataset)  # adds dataset.metadata["calibrated_confidence"]
print(dataset.metadata["calibrated_confidence"].describe())

In [ ]:
#@title Estimate FDR from the calibrated confidence (label-free)
fdr = NonParametricFDRControl()
fdr.fit(dataset.metadata["calibrated_confidence"])
md = fdr.add_psm_fdr(dataset.metadata, confidence_col="calibrated_confidence")
md = fdr.add_psm_q_value(md, confidence_col="calibrated_confidence")

for thr in (0.05, 0.10, 0.25):
    try:
        print(f"{int(thr*100)}% FDR -> calibrated-confidence cutoff {fdr.get_confidence_cutoff(thr):.3f}")
    except Exception:
        print(f"{int(thr*100)}% FDR -> not reachable on this small, low-quality subset")
print("PSMs passing 5% FDR:", int((md["psm_q_value"] <= 0.05).sum()))

o = md.sort_values("calibrated_confidence")
plt.figure(figsize=(7, 4))
plt.plot(o["calibrated_confidence"], o["psm_fdr"], marker=".")
plt.xlabel("calibrated confidence"); plt.ylabel("estimated PSM FDR")
plt.title("winnow: estimated FDR vs calibrated confidence"); plt.grid(True); plt.show()